In [ ]:
# ============================================================
# LOCAL AI MODEL — MYBINDER / JOVYAN / NO ROOT
# ============================================================

%pip install --user -q transformers accelerate torch sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Small model suitable for a normal Binder environment.
# Change this to another Hugging Face model if you have enough RAM.
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Automatically use GPU if one is available.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Loading:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if DEVICE == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32
    ).to(DEVICE)

model.eval()

print("Model loaded successfully!")
print("=" * 60)


# ============================================================
# CHAT FUNCTION
# ============================================================

def chat(prompt, max_new_tokens=256):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


# ============================================================
# INTERACTIVE LOCAL CHAT
# ============================================================

print("Local AI is ready.")
print("Type 'exit' to stop.")
print()

while True:
    prompt = input("You: ")

    if prompt.lower().strip() in ["exit", "quit", "q"]:
        print("Goodbye!")
        break

    try:
        answer = chat(prompt)
        print("\nAI:", answer)
        print()

    except Exception as e:
        print("\nERROR:", e)
        print()